## Evaluating Fine-Tuned Model

In [28]:
import json

# Load predictions
with open('data/outputs/finetuned_model_predictions.json', 'r') as f:
    predictions = json.load(f)

def evaluate_prediction(pred):
    gt = pred["ground_truth"]
    out = pred["parsed_output"]
    score = 0

    # Department (0.3)
    if out.get("department") == gt.get("department"):
        score += 0.3

    # Symptoms (0.3)
    gt_symptoms = [s.lower() for s in gt.get("symptoms", [])]
    out_symptoms = [s.lower() for s in out.get("symptoms", [])]
    if gt_symptoms:
        matches = sum(1 for s in gt_symptoms if s in out_symptoms)
        if matches == len(gt_symptoms):
            score += 0.3
        else:
            score += (matches / len(gt_symptoms)) * 0.3

    # Sentiment (0.2)
    if out.get("sentiment") == gt.get("sentiment"):
        score += 0.2

    # Urgency level (0.2)
    if out.get("urgency_level") == gt.get("urgency_level"):
        score += 0.2

    return round(score, 4)

# Evaluate all predictions
results = []
for pred in predictions:
    s = evaluate_prediction(pred)
    results.append({
        "id": pred["id"],
        "score": s,
        "valid_json": pred.get("valid_json", False)
    })

scores = [r["score"] for r in results]
avg_score = sum(scores) / len(scores)

print(f"Total predictions: {len(scores)}")
print(f"Average score: {avg_score:.4f}")
print(f"Min score: {min(scores):.4f}")
print(f"Max score: {max(scores):.4f}")
print(f"\nPer-prediction scores:")
for r in results:
    print(f"  ID {r['id']}: {r['score']:.4f}  (valid JSON: {r['valid_json']})")

Total predictions: 760
Average score: 0.7320
Min score: 0.0000
Max score: 1.0000

Per-prediction scores:
  ID 0: 1.0000  (valid JSON: True)
  ID 1: 0.6500  (valid JSON: True)
  ID 2: 0.8000  (valid JSON: True)
  ID 3: 0.0000  (valid JSON: True)
  ID 4: 0.7000  (valid JSON: True)
  ID 5: 0.4000  (valid JSON: True)
  ID 6: 0.5000  (valid JSON: True)
  ID 7: 0.7000  (valid JSON: True)
  ID 8: 0.7000  (valid JSON: True)
  ID 9: 0.7000  (valid JSON: True)
  ID 10: 1.0000  (valid JSON: True)
  ID 11: 0.6000  (valid JSON: True)
  ID 12: 0.5000  (valid JSON: True)
  ID 13: 1.0000  (valid JSON: True)
  ID 14: 1.0000  (valid JSON: True)
  ID 15: 0.7000  (valid JSON: True)
  ID 16: 1.0000  (valid JSON: True)
  ID 17: 0.6500  (valid JSON: True)
  ID 18: 0.8500  (valid JSON: True)
  ID 19: 0.7000  (valid JSON: True)
  ID 20: 0.5000  (valid JSON: True)
  ID 21: 0.8000  (valid JSON: True)
  ID 22: 1.0000  (valid JSON: True)
  ID 23: 0.6500  (valid JSON: True)
  ID 24: 1.0000  (valid JSON: True)
  ID 

## Evaluating Base Model (added explanations for some so removing that first before evaluating)

In [29]:
import json
import re

# Load base model predictions
with open('data/outputs/base_model_predictions.json', 'r') as f:
    base_predictions = json.load(f)

def parse_output(raw):
    if not raw:
        return None
    # Extract the first JSON object found in the output
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            return None
    return None

# Check nulls before re-parsing
null_before = sum(1 for p in base_predictions if p["parsed_output"] is None)
print(f"Null entries BEFORE re-parsing: {null_before}/{len(base_predictions)}")

# Re-parse null entries
for p in base_predictions:
    if p["parsed_output"] is None:
        p["parsed_output"] = parse_output(p["raw_model_output"])

# Check nulls after re-parsing
null_after = sum(1 for p in base_predictions if p["parsed_output"] is None)
print(f"Null entries AFTER re-parsing:  {null_after}/{len(base_predictions)}")
print(f"Recovered by re-parsing:        {null_before - null_after}")

def evaluate_prediction(pred):
    gt = pred["ground_truth"]
    out = pred["parsed_output"]

    if out is None:
        return 0.0

    score = 0

    # Department (0.3)
    if out.get("department") == gt.get("department"):
        score += 0.3

    # Symptoms (0.3)
    gt_symptoms = [s.lower() for s in gt.get("symptoms", [])]
    out_symptoms = [s.lower() for s in out.get("symptoms", [])]
    if gt_symptoms:
        matches = sum(1 for s in gt_symptoms if s in out_symptoms)
        if matches == len(gt_symptoms):
            score += 0.3
        else:
            score += (matches / len(gt_symptoms)) * 0.3

    # Sentiment (0.2)
    if out.get("sentiment") == gt.get("sentiment"):
        score += 0.2

    # Urgency level (0.2)
    if out.get("urgency_level") == gt.get("urgency_level"):
        score += 0.2

    return round(score, 4)

# Evaluate all predictions
base_results = []
for pred in base_predictions:
    s = evaluate_prediction(pred)
    base_results.append({
        "id": pred["id"],
        "score": s,
        "valid_json": pred.get("valid_json", False)
    })

base_scores = [r["score"] for r in base_results]
base_avg = sum(base_scores) / len(base_scores)

print(f"\nTotal predictions: {len(base_scores)}")
print(f"Average score:     {base_avg:.4f}")
print(f"Min score:         {min(base_scores):.4f}")
print(f"Max score:         {max(base_scores):.4f}")
print(f"\nPer-prediction scores:")
for r in base_results:
    print(f"  ID {r['id']}: {r['score']:.4f}  (valid JSON: {r['valid_json']})")

Null entries BEFORE re-parsing: 1/760
Null entries AFTER re-parsing:  0/760
Recovered by re-parsing:        1

Total predictions: 760
Average score:     0.3388
Min score:         0.0000
Max score:         1.0000

Per-prediction scores:
  ID 0: 0.5000  (valid JSON: True)
  ID 1: 0.4500  (valid JSON: True)
  ID 2: 0.5000  (valid JSON: True)
  ID 3: 0.3000  (valid JSON: True)
  ID 4: 0.4500  (valid JSON: True)
  ID 5: 0.2000  (valid JSON: True)
  ID 6: 0.2000  (valid JSON: True)
  ID 7: 0.3000  (valid JSON: True)
  ID 8: 0.5000  (valid JSON: True)
  ID 9: 0.0000  (valid JSON: True)
  ID 10: 0.2000  (valid JSON: True)
  ID 11: 0.5000  (valid JSON: True)
  ID 12: 0.0000  (valid JSON: True)
  ID 13: 0.3000  (valid JSON: True)
  ID 14: 0.8000  (valid JSON: True)
  ID 15: 0.5000  (valid JSON: True)
  ID 16: 0.8000  (valid JSON: True)
  ID 17: 0.4500  (valid JSON: True)
  ID 18: 0.4500  (valid JSON: True)
  ID 19: 0.0000  (valid JSON: True)
  ID 20: 0.2000  (valid JSON: True)
  ID 21: 0.6000  (

## Specific case that made the reparsing necessary

In [15]:
for p in base_predictions:
    if p["parsed_output"] is None:
        print(f"ID {p['id']}: {p['raw_model_output']}")
        print("---")

ID 570: {
  "department": "Obstetrics & Gynecology",
  "symptoms": [],
  "condition": "Pregnancy Testing Methods",
  "sentiment": "Informative",
  "urgency_level": "Low"
}

Explanation:
- department: The patient's inquiry is related to obstetrics and gynecology, so this department is appropriate.
- symptoms: Since the question does not involve any personal health issues or symptoms, an empty array is used for this key.
- condition: The condition is the topic of the patient's inquiry, which is the differences between digital and non-digital pregnancy tests.
- sentiment: The sentiment is determined by the informational nature of the patient's question.
- urgency_level: Given that the question is asking for general information rather than seeking immediate assistance, the urgency level is set as low.
---


## Specifically Urgency Level

In [21]:
gt_urgencies = set(p["ground_truth"].get("urgency_level") for p in base_predictions)
out_urgencies = set(p["parsed_output"].get("urgency_level") for p in base_predictions if p["parsed_output"] is not None)

print("Ground truth unique urgency levels:", gt_urgencies)
print("Model output unique urgency levels:", out_urgencies)

Ground truth unique urgency levels: {'High', 'Medium', 'Low'}
Model output unique urgency levels: {'Non-emergency', 'moderate', 'low', 'high', 'Low', 'Moderate', 'Non-Emergency', 'High', 'Medium'}


In [22]:
ft_urgencies = set(p["parsed_output"].get("urgency_level") for p in predictions if p["parsed_output"] is not None)
gt_urgencies = set(p["ground_truth"].get("urgency_level") for p in predictions)

print("Ground truth unique urgency levels:", gt_urgencies)
print("Fine-tuned model unique urgency levels:", ft_urgencies)

Ground truth unique urgency levels: {'High', 'Medium', 'Low'}
Fine-tuned model unique urgency levels: {'Non-emergency', 'moderate', 'low', 'high', 'Low', 'Moderate', 'Non-Emergency', 'High', 'Medium'}


In [32]:
URGENCY_NORMALIZE = {
    "low": "Low",
    "medium": "Medium",
    "moderate": "Medium",
    "high": "High",
    "critical": "Critical",
    "non-emergency": "Low",
}

def normalize_urgency(value):
    if value is None:
        return None
    return URGENCY_NORMALIZE.get(value.lower().strip(), value)

URGENCY_ORDER = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}

def urgency_analysis(preds, model_name):
    lower, higher, wrong_other = 0, 0, 0

    for pred in preds:
        gt = pred["ground_truth"]
        out = pred["parsed_output"]

        if out is None:
            continue

        gt_urgency = normalize_urgency(gt.get("urgency_level"))
        out_urgency = normalize_urgency(out.get("urgency_level"))

        # Skip correct predictions
        if gt_urgency == out_urgency:
            continue

        gt_rank = URGENCY_ORDER.get(gt_urgency)
        out_rank = URGENCY_ORDER.get(out_urgency)

        if gt_rank is None or out_rank is None:
            wrong_other += 1
        elif out_rank < gt_rank:
            lower += 1
        elif out_rank > gt_rank:
            higher += 1

    total_wrong = lower + higher + wrong_other
    print(f" {model_name} ")
    print(f"Total wrong urgency predictions: {total_wrong}")
    print(f"  Predicted LOWER than ground truth:  {lower}")
    print(f"  Predicted HIGHER than ground truth: {higher}")
    if wrong_other:
        print(f"  Unexpected/unrecognised values:     {wrong_other}")
    print()

urgency_analysis(base_predictions, "Base Model")
urgency_analysis(predictions, "Fine-tuned Model")

 Base Model 
Total wrong urgency predictions: 270
  Predicted LOWER than ground truth:  7
  Predicted HIGHER than ground truth: 263

 Fine-tuned Model 
Total wrong urgency predictions: 67
  Predicted LOWER than ground truth:  36
  Predicted HIGHER than ground truth: 31



## Summary of all experiments (with added normalization for urgency)

In [33]:
import json
import re

# Load predictions
with open('data/outputs/finetuned_model_predictions.json', 'r') as f:
    predictions = json.load(f)

with open('data/outputs/base_model_predictions.json', 'r') as f:
    base_predictions = json.load(f)

# Normalization
URGENCY_NORMALIZE = {
    "low": "Low",
    "medium": "Medium",
    "moderate": "Medium",
    "high": "High",
    "critical": "Critical",
    "non-emergency": "Low",
}

def normalize_urgency(value):
    if value is None:
        return None
    return URGENCY_NORMALIZE.get(value.lower().strip(), value)

# Re-parsing for base model nulls
def parse_output(raw):
    if not raw:
        return None
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            return None
    return None

null_before = sum(1 for p in base_predictions if p["parsed_output"] is None)
print(f"Base model null entries BEFORE re-parsing: {null_before}/{len(base_predictions)}")

for p in base_predictions:
    if p["parsed_output"] is None:
        p["parsed_output"] = parse_output(p["raw_model_output"])

null_after = sum(1 for p in base_predictions if p["parsed_output"] is None)
print(f"Base model null entries AFTER re-parsing:  {null_after}/{len(base_predictions)}")
print(f"Recovered by re-parsing:                   {null_before - null_after}\n")

# Evaluation
def evaluate_prediction(pred):
    gt = pred["ground_truth"]
    out = pred["parsed_output"]

    if out is None:
        return 0.0

    score = 0

    # Department (0.3)
    if out.get("department") == gt.get("department"):
        score += 0.3

    # Symptoms (0.3)
    gt_symptoms = [s.lower() for s in gt.get("symptoms", [])]
    out_symptoms = [s.lower() for s in out.get("symptoms", [])]
    if gt_symptoms:
        matches = sum(1 for s in gt_symptoms if s in out_symptoms)
        if matches == len(gt_symptoms):
            score += 0.3
        else:
            score += (matches / len(gt_symptoms)) * 0.3

    # Sentiment (0.2)
    if out.get("sentiment") == gt.get("sentiment"):
        score += 0.2

    # Urgency level (0.2) — normalized
    if normalize_urgency(out.get("urgency_level")) == normalize_urgency(gt.get("urgency_level")):
        score += 0.2

    return round(score, 4)

# Scoring
def run_evaluation(preds, model_name):
    results = []
    for pred in preds:
        s = evaluate_prediction(pred)
        results.append({
            "id": pred["id"],
            "score": s,
            "valid_json": pred.get("valid_json", False)
        })
    scores = [r["score"] for r in results]
    avg = sum(scores) / len(scores)
    print(f"--- {model_name} ---")
    print(f"Total predictions: {len(scores)}")
    print(f"Average score:     {avg:.4f}")
    print(f"Min score:         {min(scores):.4f}")
    print(f"Max score:         {max(scores):.4f}\n")
    return results

base_results = run_evaluation(base_predictions, "Base Model")
ft_results   = run_evaluation(predictions, "Fine-tuned Model")

# Urgency direction analysis
URGENCY_ORDER = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}

def urgency_analysis(preds, model_name):
    lower, higher, wrong_other = 0, 0, 0
    for pred in preds:
        gt  = pred["ground_truth"]
        out = pred["parsed_output"]
        if out is None:
            continue
        gt_urgency  = normalize_urgency(gt.get("urgency_level"))
        out_urgency = normalize_urgency(out.get("urgency_level"))
        if gt_urgency == out_urgency:
            continue
        gt_rank  = URGENCY_ORDER.get(gt_urgency)
        out_rank = URGENCY_ORDER.get(out_urgency)
        if gt_rank is None or out_rank is None:
            wrong_other += 1
        elif out_rank < gt_rank:
            lower += 1
        elif out_rank > gt_rank:
            higher += 1

    total_wrong = lower + higher + wrong_other
    print(f"{model_name} Urgency Analysis")
    print(f"Total wrong urgency predictions: {total_wrong}")
    print(f"  Predicted LOWER than ground truth:  {lower}")
    print(f"  Predicted HIGHER than ground truth: {higher}")
    if wrong_other:
        print(f"  Unexpected/unrecognised values:     {wrong_other}")
    print()

urgency_analysis(base_predictions, "Base Model")
urgency_analysis(predictions, "Fine-tuned Model")

# Comparison
base_avg = sum(r["score"] for r in base_results) / len(base_results)
ft_avg   = sum(r["score"] for r in ft_results)   / len(ft_results)
print("Model Comparison")
print(f"Base model avg score:       {base_avg:.4f}")
print(f"Fine-tuned model avg score: {ft_avg:.4f}")
print(f"Improvement:                {ft_avg - base_avg:+.4f}")

Base model null entries BEFORE re-parsing: 1/760
Base model null entries AFTER re-parsing:  0/760
Recovered by re-parsing:                   1

--- Base Model ---
Total predictions: 760
Average score:     0.4246
Min score:         0.0000
Max score:         1.0000

--- Fine-tuned Model ---
Total predictions: 760
Average score:     0.7320
Min score:         0.0000
Max score:         1.0000

Base Model Urgency Analysis
Total wrong urgency predictions: 270
  Predicted LOWER than ground truth:  7
  Predicted HIGHER than ground truth: 263

Fine-tuned Model Urgency Analysis
Total wrong urgency predictions: 67
  Predicted LOWER than ground truth:  36
  Predicted HIGHER than ground truth: 31

Model Comparison
Base model avg score:       0.4246
Fine-tuned model avg score: 0.7320
Improvement:                +0.3074
